# Repeat runs A: runs 2 and 3 of the persona + norms prompt (GPT-6 Luna, 500 ads, no guidelines)
Countries (first batch of 12): United States, Germany, South Korea, Pakistan, United Arab Emirates, Belgium, Bahrain, Morocco, Austria, Bulgaria, Bangladesh, Ghana.
Same prompt, model and settings as the first run (`ng_norms`, prompt version printed in R1 must match the first
run's); only `repeat` = 2 and 3 are sent. 12 countries x 2 runs x 500 ads = **12,000 requests**, normal requests in
parallel, saved as they finish (resumable).
**Order:** R0 -> R1 (check the printed prompts) -> R3a -> R5. Inputs: the 500-ad prep output (same as before).
Settings: Internet on; secret `OPENAI_API_KEY`. No GPU needed.

In [ ]:
# R0. Setup
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "openai"], check=True)

import base64, datetime as dt, glob, hashlib, json, re, time
from pathlib import Path
import pandas as pd
from openai import OpenAI
import openai

MODELS = ["gpt-6-luna"]
BATCH_MODELS = []                  # Batch API off: the first attempt's batches sat "in progress" for 11 h
SYNC_MODELS = ["gpt-6-luna"]       # normal requests; R3a first cancels this project's running batches and keeps any answers they finished
# To finish stuck Luna batches as normal requests instead: BATCH_MODELS = [] and SYNC_MODELS = ["gpt-6.1-terra", "gpt-6-luna"]
# (R3a then cancels the running Luna batches, keeps everything they already answered, and sends only the rest)
SYNC_WORKERS = 12                  # parallel requests for SYNC_MODELS (the SDK backs off on rate-limit errors)
PRICE = {"gpt-6-luna": (0.10, 0.01, 0.50), "gpt-6.1-terra": (1.00, 0.05, 5.00)}  # $/1M: input, cached input, output
REASONING_EFFORT = "none"          # fixed and recorded; "none" = no billed thinking tokens
IMAGE_DETAIL = "low"               # set after the pilot (R2 compares "low" and "auto")
MAX_OUTPUT_TOKENS = 300            # replies are ~50-90 tokens; a lower cap also lowers the batch queue estimate
PROJECT = "persona-run"  # tag used to find these batches again
WAVE = "openai-repsA"
BASELINE_RUNS = 2                  # 2 baseline runs = run-to-run noise estimate
MAX_BATCH_MB = 180                 # OpenAI limit is 200 MB per batch file
CHUNK_SIZE = {"gpt-6-luna": 300, "gpt-6.1-terra": 100}   # requests per batch file
QUEUE_LIMIT = {"gpt-6.1-terra": 1_350_000, "gpt-6-luna": 20_000_000}  # enqueued-token limit per model; learned from errors if missing
QUEUE_FILL = 0.85                  # submit while the estimated queued tokens stay below 85% of the limit
MAX_ACTIVE_PER_MODEL = 30           # submit all batch files at once; a queue-limit rejection is retried later
RUN_PILOT = False                  # the pilot has been run; set True to run it again

WORK = Path("/kaggle/working"); OUT = WORK / "openai_out"; OUT.mkdir(exist_ok=True)

def find_input(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    assert hits, f"{name} not found - attach the prep notebook's output"
    return Path(hits[0])

INPUTS = find_input("ad_inputs_500.jsonl")
FRAMES = INPUTS.parent / "frames"
GUIDE = find_input("guidelines.txt").read_text().strip()   # same file the Gemini runs used
SAMPLE = pd.read_csv(find_input("ad_sample_500.csv"), dtype={"ad_id_clean": str})
HUMAN = SAMPLE.set_index("ad_id_clean").primary_tag.to_dict()     # for the results file only, never the prompt
ADS = [json.loads(l) for l in open(INPUTS)]
assert FRAMES.is_dir() and len(ADS) == 500 and GUIDE

try:
    from kaggle_secrets import UserSecretsClient
    API_KEY = UserSecretsClient().get_secret("OPENAI_API_KEY")
except Exception:
    import os; API_KEY = os.environ["OPENAI_API_KEY"]
# answers from an earlier run of this notebook (attach that version's output): copy them in so nothing is resent
import shutil
for f in sorted(glob.glob("/kaggle/input/**/openai_out/sync_*.jsonl", recursive=True)):
    dst = OUT / Path(f).name
    if not dst.exists():
        shutil.copy(f, dst); print("restored earlier answers:", dst.name, sum(1 for _ in open(dst)), "lines")

client = OpenAI(api_key=API_KEY, max_retries=5, timeout=180)
sync_client = OpenAI(api_key=API_KEY, max_retries=10, timeout=180)   # more patient with rate limits
print("openai", openai.__version__, "| ads", len(ADS), "| frames dir", FRAMES)

## R1. Prompts and request bodies

In [ ]:
# 12 held-out countries (none in the original 12): 3 per COSI 2023 quartile, Muslim-majority (MM) and others mixed
# within each quartile (A 1+2, B 2+1, C 1+2, D 2+1), so the MM comparison is not confounded with COSI quartile
COUNTRIES = ['United States', 'Germany', 'South Korea', 'Pakistan', 'United Arab Emirates', 'Belgium', 'Bahrain', 'Morocco', 'Austria', 'Bulgaria', 'Bangladesh', 'Ghana']
assert len(COUNTRIES) == 12
# same wording as the first run: v4 wrote "the United States" / "the United Arab Emirates"
COUNTRY_PHRASE = {"United States": "the United States", "United Arab Emirates": "the United Arab Emirates"}
TEMPLATE_OF = {"g_plain": "G", "g_norms": "G", "ng_plain": "NG", "ng_norms": "NG"}   # unchanged -> same PROMPT_VERSION
CONDITIONS = [("ng_norms", c, r) for c in COUNTRIES for r in (2, 3)]
assert len(CONDITIONS) == 24
def persona(attr, value):
    if value is None:
        return "You are a content moderator."
    place = COUNTRY_PHRASE.get(value, value)
    p = f"You are a content moderator from {place}. You're moderating advertisements for audiences in {place}."
    if attr == "g_norms":
        p += f" Apply the guidelines below according to the cultural norms and values of {place}."
    else:
        p += f" Judge the ads according to the cultural norms and values of {place}."
    return p

TEMPLATES = {"G": """{persona}

Your task is to review video advertisements that may be shown on YouTube videos labelled "Made for Kids", which are made for and watched mainly by young children (under 13). For each ad you receive its YouTube metadata, an automatic transcript of its audio, and still frames sampled evenly across the whole ad. Decide whether it is appropriate to show on Made for Kids content according to the advertising guidelines below.

<guidelines>
{guidelines}
</guidelines>

Assign exactly one label:
- Inappropriate: the ad contains content that is restricted or prohibited under the guidelines above.
- Child-directed: the ad contains no restricted or prohibited content and has features or appeals specifically targeted at children under 13 (e.g., child-oriented toys, cartoon characters, or child-rated movies).
- Irrelevant: the ad contains no restricted or prohibited content and promotes products or services intended primarily for audiences aged 13 and above.

If the label is Inappropriate, list every restricted category that applies; otherwise return an empty list. Give a one- to two-sentence rationale based on what is shown, said or described in the ad.""", "NG": """{persona}

Your task is to review video advertisements that may be shown on YouTube videos labelled "Made for Kids", which are made for and watched mainly by young children (under 13). For each ad you receive its YouTube metadata, an automatic transcript of its audio, and still frames sampled evenly across the whole ad. Decide whether it is appropriate to show on Made for Kids content.

Assign exactly one label:
- Inappropriate: the ad is not appropriate to show to young children on Made for Kids content.
- Child-directed: the ad is appropriate for young children and has features or appeals specifically targeted at children under 13 (e.g., child-oriented toys, cartoon characters, or child-rated movies).
- Irrelevant: the ad is appropriate for young children but promotes products or services intended primarily for audiences aged 13 and above.

If the label is Inappropriate, list briefly (a few words each) what makes it inappropriate; otherwise return an empty list. Give a one- to two-sentence rationale based on what is shown, said or described in the ad.""", "CATCHALL": """{persona}

Your task is to review video advertisements that may be shown on YouTube videos labelled "Made for Kids", which are made for and watched mainly by young children (under 13). For each ad you receive its YouTube metadata, an automatic transcript of its audio, and still frames sampled evenly across the whole ad. Decide whether it is appropriate to show on Made for Kids content according to the advertising guidelines below.

<guidelines>
{guidelines}
</guidelines>

Assign exactly one label:
- Inappropriate: the ad contains content that is restricted or prohibited under the guidelines above, or is otherwise not appropriate to show to young children.
- Child-directed: the ad contains no restricted or prohibited content and has features or appeals specifically targeted at children under 13 (e.g., child-oriented toys, cartoon characters, or child-rated movies).
- Irrelevant: the ad contains no restricted or prohibited content and promotes products or services intended primarily for audiences aged 13 and above.

If the label is Inappropriate, list every restricted category that applies, and add "Other (not covered by the guidelines)" if part of the reason is not covered by any category; otherwise return an empty list. Give a one- to two-sentence rationale based on what is shown, said or described in the ad."""}
CATEGORIES = [
    "Alcohol", "Tobacco", "Recreational Drugs", "Gambling and Casino Games", "Teen and Adult Media",
    "Dangerous Content", "Complex Speculative Financial Products", "Beauty and Cosmetics",
    "Body Modification and Weight Loss", "Food and Beverage", "Health and Wellness",
    "Pharmaceuticals and Supplements", "Contests and Sweepstakes", "Mobile Subscriptions", "Social Networks",
    "Spying and Arrest Records", "Quizzes", "Video Game Skins / Loot Boxes", "Virtual Worlds / Adult Chat Rooms",
    "Astrology, Occult, Paranormal", "Politics, Religion, Sensitive Social Issues",
    "Adult / Sexually Suggestive Content", "Dating and Relationships", "Romantic Content",
    "Significant Skin Exposure", "Fight Sports and Martial Arts", "First-Person Shooters / Battle Games",
    "Shocking or Scary Content"]
def _schema(items):
    return {"type": "object", "additionalProperties": False, "required": ["label", "categories", "rationale"],
            "properties": {"label": {"type": "string", "enum": ["Inappropriate", "Child-directed", "Irrelevant"]},
                           "categories": {"type": "array", "items": items}, "rationale": {"type": "string"}}}
SCHEMAS = {"G": _schema({"type": "string", "enum": CATEGORIES}),
           "NG": _schema({"type": "string"}),                                   # free text: no policy vocabulary
           "CATCHALL": _schema({"type": "string", "enum": CATEGORIES + ["Other (not covered by the guidelines)"]})}
def template_for(attr):
    return TEMPLATES[TEMPLATE_OF[attr]]
def schema_for(attr):
    return SCHEMAS[TEMPLATE_OF[attr]]
def condition_id(attr, value, rep):
    return f"{attr}#r{rep}" if value is None else f"{attr}={value}#r{rep}"
SCHEMA = SCHEMAS["G"]                       # label enum used when parsing answers (same in every schema)
SYSTEM_TEMPLATE = "".join(TEMPLATES[TEMPLATE_OF[a]] for a in sorted(TEMPLATE_OF))   # for PROMPT_VERSION only
PROMPT_VERSION = hashlib.sha1((SYSTEM_TEMPLATE + GUIDE + json.dumps(SCHEMAS) + json.dumps(TEMPLATE_OF) + IMAGE_DETAIL)
                              .encode()).hexdigest()[:8]

_img_cache = {}
def img_url(fname):
    if fname not in _img_cache:
        _img_cache[fname] = "data:image/jpeg;base64," + base64.b64encode((FRAMES / fname).read_bytes()).decode()
    return _img_cache[fname]

def user_content(ad, detail=IMAGE_DETAIL):
    parts = [{"type": "input_text", "text": ad["context"]}]
    for f in ad["frames"]:
        parts.append({"type": "input_text", "text": f["label"] + ":"})
        parts.append({"type": "input_image", "image_url": img_url(f["file"]), "detail": detail})
    parts.append({"type": "input_text", "text": "Review this advertisement."})
    return parts

def body(model, attr, value, ad, detail=IMAGE_DETAIL):
    return {"model": model,
            "instructions": template_for(attr).format(persona=persona(attr, value), guidelines=GUIDE),
            "input": [{"role": "user", "content": user_content(ad, detail)}],
            "text": {"format": {"type": "json_schema", "name": "verdict", "schema": schema_for(attr), "strict": True}},
            "reasoning": {"effort": REASONING_EFFORT},
            "max_output_tokens": MAX_OUTPUT_TOKENS,
            "store": False}

def custom_id(model, attr, value, rep, ad_id):
    return f"{model}|{condition_id(attr, value, rep)}|{ad_id}"

def parse_response(resp):
    """resp: Responses API body (dict). Returns (verdict dict or None, raw text, refusal)."""
    text, refusal = "", None
    for item in resp.get("output", []) or []:
        if item.get("type") == "message":
            for c in item.get("content", []) or []:
                if c.get("type") == "output_text":
                    text += c.get("text", "")
                elif c.get("type") == "refusal":
                    refusal = c.get("refusal")
    try:
        v = json.loads(text)
        ok = v.get("label") in SCHEMA["properties"]["label"]["enum"] and isinstance(v.get("categories"), list)
        return (v if ok else None), text, refusal
    except Exception:
        return None, text, refusal

ACTIVE = {"validating", "in_progress", "finalizing"}

def our_batches():
    """Latest batch per chunk for this project and prompt version."""
    out = {}
    for b in client.batches.list(limit=100):
        m = b.metadata or {}
        if m.get("project") == PROJECT and m.get("prompt_version") == PROMPT_VERSION:
            prev = out.get(m.get("chunk"))
            if prev is None or b.created_at > prev.created_at:
                out[m.get("chunk")] = b
    return out

print("prompt_version", PROMPT_VERSION, "| conditions", len(CONDITIONS), "| requests", len(CONDITIONS) * len(ADS))
print("\nPersona lines:", *[persona("ng_norms", c) for c in COUNTRIES[:3]], sep="\n  ")
print("\n===== ng_norms full prompt (Japan) =====\n" + template_for("ng_norms").format(persona=persona("ng_norms", "Japan"), guidelines=""))

## R2. Pilot (synchronous, a few cents)
3 ads (shortest, median, longest) × both models × image detail `low` and `auto`, baseline persona.
Shows the real tokens per request and projects the cost of the full batch run (Batch = 50% off).
**Check the projected cost and the parsed outputs before going on.** Then set `IMAGE_DETAIL` in R0 if needed
(changing it changes `PROMPT_VERSION`) and re-run R0–R1.

In [ ]:
def run_pilot():
    pilot_ads = sorted(ADS, key=lambda a: a["duration"])
    pilot_ads = [pilot_ads[0], pilot_ads[len(pilot_ads) // 2], pilot_ads[-1]]
    rows = []
    for model in MODELS:
        for detail in ("low", "auto"):
            for ad in pilot_ads:
                t0 = time.time()
                try:
                    r = client.responses.create(**body(model, "baseline", None, ad, detail)).model_dump()
                    v, text, refusal = parse_response(r)
                    u = r.get("usage") or {}
                    rows.append({"model": model, "detail": detail, "ad": ad["ad_id"], "frames": len(ad["frames"]),
                                 "in_tok": u.get("input_tokens"), "out_tok": u.get("output_tokens"),
                                 "reasoning_tok": (u.get("output_tokens_details") or {}).get("reasoning_tokens"),
                                 "label": v and v["label"], "categories": v and v["categories"],
                                 "rationale": (v or {}).get("rationale", text)[:160], "refusal": refusal,
                                 "status": r.get("status"), "secs": round(time.time() - t0, 1)})
                except Exception as e:
                    rows.append({"model": model, "detail": detail, "ad": ad["ad_id"], "error": repr(e)[:300]})
    PILOT = pd.DataFrame(rows)
    display(PILOT)

    n_req = len(CONDITIONS) * len(ADS)
    est = (PILOT.dropna(subset=["in_tok"]).groupby(["model", "detail"])[["in_tok", "out_tok"]].mean()
           .assign(full_run_usd_batch=lambda d: [n_req * (i * PRICE[m][0] + o * PRICE[m][2]) / 1e6 / 2
                                                 for (m, _), i, o in zip(d.index, d.in_tok, d.out_tok)]))
    print(f"Projected cost of one full run ({n_req} requests) per model, Batch price, "
          "from these 3 ads (long ads cost more, so treat as rough):")
    display(est.round(3))

if RUN_PILOT:
    run_pilot()
else:
    print("pilot skipped (RUN_PILOT = False)")

## R3a. Normal (non-batch) requests for `SYNC_MODELS` (runs first; independent of the batches)
First a single request reads your account's rate limits from the response headers. Then all remaining
requests run in parallel (`SYNC_WORKERS`), written to `openai_out/sync_<model>.jsonl` as they finish.
**Resumable:** re-running skips everything already answered (including any answers from earlier batches).

In [ ]:
COND_INFO = {condition_id(a, v, r): (a, v, r, persona(a, v)) for a, v, r in CONDITIONS}

def base_record(model, cond, ad_id, ts, method, batch_id=None):
    attr, value, rep, pers = COND_INFO[cond]
    return {"wave": WAVE, "condition_id": cond, "attribute": attr, "value": value, "repeat": rep,
            "persona": pers, "ad_id": ad_id, "primary_tag": HUMAN.get(ad_id), "mode": "frames+transcript",
            "coverage": 1.0, "model": model, "thinking_level": REASONING_EFFORT, "prompt_version": PROMPT_VERSION,
            "key": None, "method": method, "batch_id": batch_id, "ts_utc": ts}

def fill_from_response(rec, rb):
    v, text, refusal = parse_response(rb)
    u = rb.get("usage") or {}
    rec.update(status="ok" if v else ("refused" if refusal else "unparsed"),
               label=v and v["label"], categories=v and v["categories"], rationale=v and v["rationale"],
               raw_text=None if v else (refusal or text), finish_reason=rb.get("status"), error=None,
               prompt_tokens=u.get("input_tokens"), output_tokens=u.get("output_tokens"),
               thought_tokens=(u.get("output_tokens_details") or {}).get("reasoning_tokens"),
               cached_tokens=(u.get("input_tokens_details") or {}).get("cached_tokens"))
    return rec

def fill_error(rec, err):
    rec.update(status="error", label=None, categories=None, rationale=None, raw_text=None, error=str(err)[:300])
    return rec

def all_our_batches():
    """Every batch of this project and prompt version, including superseded attempts."""
    return [b for b in client.batches.list(limit=100)
            if (b.metadata or {}).get("project") == PROJECT
            and (b.metadata or {}).get("prompt_version") == PROMPT_VERSION]

def batch_records():
    out = []
    for b in all_our_batches():                     # all attempts: none of their answers are lost
        if b.status not in ("completed", "cancelled", "expired"):
            continue
        ts = dt.datetime.fromtimestamp(b.completed_at or b.created_at, dt.timezone.utc).isoformat()
        for fid, kind in ((b.output_file_id, "out"), (b.error_file_id, "err")):
            if not fid:
                continue
            for line in client.files.content(fid).text.splitlines():
                j = json.loads(line)
                model, cond, ad_id = j["custom_id"].split("|")
                rec = base_record(model, cond, ad_id, ts, "batch", b.id)
                resp = j.get("response") or {}
                rb = resp.get("body") or {}
                if kind == "out" and resp.get("status_code") == 200:
                    fill_from_response(rec, rb)
                else:
                    fill_error(rec, json.dumps(j.get("error") or rb.get("error") or resp))
                out.append(rec)
    return out

def sync_file(model):
    return OUT / f"sync_{model}_{WAVE}_{PROMPT_VERSION}.jsonl"

def sync_records(model):
    f = sync_file(model)
    recs = [json.loads(l) for l in open(f)] if f.exists() else []
    return [x for x in recs if x.get("prompt_version") == PROMPT_VERSION and x.get("wave") == WAVE]

import random, threading
from concurrent.futures import ThreadPoolExecutor, as_completed

# Terra batches must not run alongside the normal requests: cancel any still running and wait until they settle,
# so their finished answers are counted below (nothing is paid twice)
for _ in range(30):
    live = [b for b in all_our_batches() if (b.metadata or {}).get("model") in SYNC_MODELS
            and b.status in ACTIVE | {"cancelling"}]
    if not live:
        break
    for b in live:
        if b.status in ACTIVE:
            client.batches.cancel(b.id); print("cancelled", (b.metadata or {}).get("chunk"), b.id)
    time.sleep(20)

for model in SYNC_MODELS:
    # 1. rate limits (one real request; its answer is kept)
    raw = sync_client.responses.with_raw_response.create(**body(model, *CONDITIONS[0][:2], ADS[0]))
    hdr = {k: raw.headers.get(k) for k in ("x-ratelimit-limit-requests", "x-ratelimit-limit-tokens",
                                           "x-ratelimit-remaining-requests", "x-ratelimit-remaining-tokens")}
    print(model, "rate limits:", hdr)
    workers = SYNC_WORKERS
    try:                                   # ~4,500 tokens and ~3 s per request -> ~20 requests/min per worker
        tpm_rpm = int(hdr["x-ratelimit-limit-tokens"]) / 4500
        rpm = min(tpm_rpm, int(hdr["x-ratelimit-limit-requests"]))
        workers = max(1, min(SYNC_WORKERS, int(rpm * 0.8 / 20)))
        print(f"  allows ~{rpm:.0f} requests/min -> {workers} parallel workers "
              f"(~{len(CONDITIONS) * len(ADS) / max(min(rpm * 0.8, workers * 20), 1) / 60:.1f} h for a full run)")
    except Exception:
        print(f"  limits not readable; using {workers} workers")
    first = fill_from_response(base_record(model, condition_id(*CONDITIONS[0]), ADS[0]["ad_id"],
                               dt.datetime.now(dt.timezone.utc).isoformat(), "sync"), raw.parse().model_dump())
    with open(sync_file(model), "a") as fh:
        fh.write(json.dumps(first, ensure_ascii=False, default=str) + "\n")

    # 2. everything not yet answered (by a batch or an earlier sync run)
    done = {(r["condition_id"], r["ad_id"]) for r in batch_records() + sync_records(model)
            if r["model"] == model and r["status"] == "ok"}
    todo = [(a, v, r, ad) for a, v, r in CONDITIONS for ad in ADS if (condition_id(a, v, r), ad["ad_id"]) not in done]
    random.Random(7).shuffle(todo)          # interleave conditions over time, as in the Gemini runs
    print(f"{model}: {len(done)} already answered, {len(todo)} to send with {workers} workers")

    lock, counts, t0 = threading.Lock(), {}, time.time()
    def one(a, v, r, ad):
        rec = base_record(model, condition_id(a, v, r), ad["ad_id"], dt.datetime.now(dt.timezone.utc).isoformat(), "sync")
        try:
            fill_from_response(rec, sync_client.responses.create(**body(model, a, v, ad)).model_dump())
        except Exception as e:
            fill_error(rec, repr(e))
        with lock:
            with open(sync_file(model), "a") as fh:
                fh.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")
            counts[rec["status"]] = counts.get(rec["status"], 0) + 1
            n = sum(counts.values())
            if n % 200 == 0 or n == len(todo):
                print(f"{dt.datetime.now():%H:%M} {model}: {n}/{len(todo)} {counts} "
                      f"({n / max(time.time() - t0, 1) * 60:.0f}/min)", flush=True)

    with ThreadPoolExecutor(workers) as ex:
        for f_ in as_completed([ex.submit(one, *t) for t in todo]):
            f_.result()

## R3. Build batch files (`BATCH_MODELS` only)
One JSONL file per (model, condition), split by `CHUNK_SIZE` and the 180 MB file limit.

In [ ]:
BATCH_DIR = Path("/tmp/batches"); BATCH_DIR.mkdir(exist_ok=True)   # temporary: not saved as output
def est_tokens(attr, value, ad):
    """Conservative estimate of what the batch queue counts per request (input + max output)."""
    instr = template_for(attr).format(persona=persona(attr, value), guidelines=GUIDE)
    return len(instr) // 4 + len(ad["context"]) // 2 + 170 * len(ad["frames"]) + 40 + MAX_OUTPUT_TOKENS

CHUNKS = []                                            # (model, chunk_name, path, n_requests, est_tokens)
for model in BATCH_MODELS:
    size_n = CHUNK_SIZE.get(model, 300)
    for attr, value, rep in CONDITIONS:
        base = f"{model}__{condition_id(attr, value, rep).replace('=', '-').replace(' ', '_').replace('#', '_')}"
        if size_n != 300:
            base += f"_n{size_n}"
        part, size, n, tok, fh = 0, 0, 0, 0, None
        for ad in ADS:
            line = json.dumps({"custom_id": custom_id(model, attr, value, rep, ad["ad_id"]), "method": "POST",
                               "url": "/v1/responses", "body": body(model, attr, value, ad)}) + "\n"
            if fh is None or n >= size_n or size + len(line) > MAX_BATCH_MB * 1e6:
                if fh:
                    fh.close(); CHUNKS.append((model, name, path, n, tok))
                name = f"{base}_p{part}"; path = BATCH_DIR / f"{name}.jsonl"
                fh, size, n, tok, part = open(path, "w"), 0, 0, 0, part + 1
            fh.write(line); size += len(line); n += 1; tok += est_tokens(attr, value, ad)
        fh.close(); CHUNKS.append((model, name, path, n, tok))
CH = pd.DataFrame(CHUNKS, columns=["model", "chunk", "path", "n", "est_tokens"]).assign(
    mb=lambda d: [round(p.stat().st_size / 1e6, 1) for p in d.path])
print(f"{len(CH)} batch files | requests per model: {CH.groupby('model').n.sum().to_dict()} | "
      f"largest file {CH.mb.max()} MB")
print("estimated queue tokens per batch file:", CH.groupby("model").est_tokens.max().to_dict())

## R4. Submit and wait
Keeps each model's queue within its enqueued-token limit (known limits in `QUEUE_LIMIT`; others are read
from the first rejection message), polls every 2 minutes, submits more as earlier batches finish, and stops when
all are done or after `MAX_HOURS`. A rejected batch is resubmitted later **reusing the already-uploaded file**.
Batches from an older chunking setup (not in the current file list) are cancelled. Safe to stop and re-run, also
in a new session: batches are found again through the API by their `PROJECT` and `PROMPT_VERSION` tags.

In [ ]:
MAX_HOURS = 11                     # Kaggle stops a commit at 12 h; if batches are still running, re-run in a new commit

def submit(row, reuse_file_id=None):
    if reuse_file_id:
        try:
            return client.batches.create(input_file_id=reuse_file_id, endpoint="/v1/responses",
                                         completion_window="24h",
                                         metadata={"project": PROJECT, "prompt_version": PROMPT_VERSION,
                                                   "model": row.model, "chunk": row.chunk})
        except Exception as e:
            print(f"  could not reuse {reuse_file_id} ({repr(e)[:80]}); uploading again")
    f = client.files.create(file=open(row.path, "rb"), purpose="batch")
    return client.batches.create(input_file_id=f.id, endpoint="/v1/responses", completion_window="24h",
                                 metadata={"project": PROJECT, "prompt_version": PROMPT_VERSION,
                                           "model": row.model, "chunk": row.chunk})

def limit_from_error(b):
    m = re.search(r"Limit: ([\d,]+) enqueued tokens", str(getattr(b, "errors", "")))
    return int(m.group(1).replace(",", "")) if m else None

RETRYABLE = ("not_submitted", "failed", "expired", "cancelled")
t_end = time.time() + MAX_HOURS * 3600
while True:
    bs = our_batches()
    for chunk, b in bs.items():                         # cancel batches from an older chunking setup
        if chunk not in set(CH.chunk) and b.status in ACTIVE:
            client.batches.cancel(b.id); print(f"cancelled stale batch {chunk} ({b.id})")
    status = {c: (bs[c].status if c in bs else "not_submitted") for c in CH.chunk}
    for model, g in CH.groupby("model"):
        for c in g.chunk:                               # learn the queue limit from a rejection
            if status[c] == "failed" and limit_from_error(bs[c]):
                QUEUE_LIMIT[model] = limit_from_error(bs[c])
        limit = QUEUE_LIMIT.get(model)
        act = g[[status[c] in ACTIVE for c in g.chunk]]
        queued, n_active = int(act.est_tokens.sum()), len(act)
        if n_active and any(status[c] == "failed" for c in g.chunk):
            continue        # a batch was just rejected for the queue limit: wait until a running one finishes
        for row in g.itertuples():
            if status[row.chunk] not in RETRYABLE:
                continue
            if n_active >= MAX_ACTIVE_PER_MODEL:
                break
            if limit and n_active and queued + row.est_tokens > QUEUE_FILL * limit:
                break
            prev = bs.get(row.chunk)
            try:
                b = submit(row, prev.input_file_id if prev is not None else None)
                status[row.chunk] = b.status; queued += row.est_tokens; n_active += 1
                print(f"{dt.datetime.now():%H:%M} submitted {row.chunk} ({row.n} requests, "
                      f"~{row.est_tokens/1e6:.2f}M queue tokens) -> {b.id}")
            except Exception as e:
                print(f"{dt.datetime.now():%H:%M} could not submit {row.chunk}: {repr(e)[:200]}")
                break
    counts = pd.Series(status).value_counts().to_dict()
    print(f"{dt.datetime.now():%H:%M} {counts} | queue limits {QUEUE_LIMIT}", flush=True)
    if all(s_ == "completed" for s_ in status.values()) or time.time() > t_end:
        break
    time.sleep(120)

## R5. Collect results
Combines batch answers and normal-request answers into the same format as the Gemini results
(`results_<model>.jsonl`, one file per model) and lists anything missing. Missing `SYNC_MODELS` requests: re-run
R3a. Missing `BATCH_MODELS` requests: run the retry cell below, then R4 and R5 again.

In [ ]:
records = batch_records() + [r for m in SYNC_MODELS for r in sync_records(m)]
for chunk, b in our_batches().items():
    if b.status in ACTIVE:
        print(f"still running: {chunk} ({b.status})")
R = pd.DataFrame(records)
if R.empty:                       # nothing collected yet: keep the columns so the summaries below still run
    R = pd.DataFrame(columns=["wave", "condition_id", "ad_id", "model", "method", "ts_utc", "status", "label",
                              "categories", "rationale", "prompt_tokens", "output_tokens", "cached_tokens"])
    print("no answers collected yet")
R = (R.assign(_ok=R.status.eq("ok")).sort_values(["_ok", "ts_utc"]).drop_duplicates(
        ["model", "condition_id", "ad_id"], keep="last").drop(columns="_ok"))
for model, g in R.groupby("model"):
    with open(OUT / f"results_{model}_{WAVE}.jsonl", "w") as fh:
        for rec in g.to_dict("records"):
            fh.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")

expected = {custom_id(m, a, v, r, ad["ad_id"]) for m in MODELS for a, v, r in CONDITIONS for ad in ADS}
got_ok = set(R.loc[R.status == "ok"].apply(lambda r: f"{r.model}|{r.condition_id}|{r.ad_id}", axis=1))
MISSING = sorted(expected - got_ok)
print("status by model:"); display(R.groupby(["model", "status"]).size().unstack(fill_value=0))
print(f"missing or failed requests: {len(MISSING)}")
ok = R[R.status == "ok"].copy()
for c in ("prompt_tokens", "output_tokens", "cached_tokens"):
    ok[c] = pd.to_numeric(ok[c]).fillna(0)
ok["cost_usd"] = [((r.prompt_tokens - r.cached_tokens) * PRICE[r.model][0] + r.cached_tokens * PRICE[r.model][1]
                   + r.output_tokens * PRICE[r.model][2]) / 1e6 * (0.5 if r.method == "batch" else 1.0)
                  for r in ok.itertuples()]
print("tokens and approximate cost (successful requests only):")
display(ok.groupby(["model", "method"])[["prompt_tokens", "cached_tokens", "output_tokens", "cost_usd"]].sum().round(2))
print("label distribution:"); display(R[R.status == "ok"].groupby(["model", "label"]).size().unstack(fill_value=0))

In [ ]:
# R5b. Retry: write the missing requests into a new batch file, then re-run R4 and R5
if MISSING:
    k = sum(c.startswith("retry") for c in CH.chunk)
    byid = {ad["ad_id"]: ad for ad in ADS}
    inv = {condition_id(a, v, r): (a, v) for a, v, r in CONDITIONS}
    new = []
    for model in BATCH_MODELS:
        ids = [m for m in MISSING if m.startswith(model + "|")]
        if not ids:
            continue
        name = f"retry{k}__{model}"; path = BATCH_DIR / f"{name}.jsonl"
        with open(path, "w") as fh:
            for cid in ids:
                _, cond, ad_id = cid.split("|"); a, v = inv[cond]
                fh.write(json.dumps({"custom_id": cid, "method": "POST", "url": "/v1/responses",
                                     "body": body(model, a, v, byid[ad_id])}) + "\n")
        new.append((model, name, path, len(ids), round(path.stat().st_size / 1e6, 1)))
    new = [(m, n, p, k, k * 5000, mb) for m, n, p, k, mb in new]       # rough queue estimate for retries
    CH = pd.concat([CH, pd.DataFrame(new, columns=["model", "chunk", "path", "n", "est_tokens", "mb"])],
                   ignore_index=True)
    if new:
        print("added retry batch files:", [(n[1], n[3]) for n in new], "-> re-run R4 and R5")
    if any(m.split("|")[0] in SYNC_MODELS for m in MISSING):
        print("missing normal-request answers: re-run R3a")
else:
    print("nothing to retry")

## R6. Clean up uploaded input files (optional, after everything is collected)
Deletes this project's uploaded batch input files that no running batch still needs, including duplicates
from resubmissions. Results (output files) are not touched; OpenAI deletes those after 30 days.

In [ ]:
batches = list(client.batches.list(limit=100))
in_use = {b.input_file_id for b in batches if b.status in ACTIVE}
pattern = re.compile(r"^(gpt-6[^_]*__|retry\d+__).*\.jsonl$")
deleted = 0
for f in client.files.list(purpose="batch"):
    if pattern.match(f.filename or "") and f.id not in in_use:
        client.files.delete(f.id); deleted += 1
print(f"deleted {deleted} uploaded input files; kept {len(in_use)} still used by running batches")